In [35]:
%load_ext autoreload
%autoreload 2
from pathlib import Path
import yaml
import pandas as pd
idx_slice = pd.IndexSlice
from itertools import product
import pypsa
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning) # Comment out for debugging and development
warnings.simplefilter(action='ignore', category=DeprecationWarning) # Comment out for debugging and development
import plotly.express as px

from plot_helpers import (
    init_stats_dict, 
    drop_index_levels, 
    update_layout, 
    prepare_dataframe,
    get_supply_demand_from_balance,
    save_plotly_fig)

run_name_prefix = "H2G_B_250507"
all_countries = ["MA"
               #"EG", "NA", "MA", "ZA", "KE", "ET", "CG", "TZ", "GH", "TN", "NG"
               ]
all_configs = {
    country:
        yaml.safe_load(
            Path(f"config.{country}_{run_name_prefix}.yaml").read_text()) 
        for country in all_countries
}

all_postnetworks_dir = {
    country:
        Path.cwd() / all_configs[country]["results_dir"] / f"{country}_{run_name_prefix}" / "postnetworks"
        for country in all_countries
}

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [36]:
# NB: the following code only works if one country is selected per run.
all_wildcards = {
    country: 
    {
        "run_name": [], 
        "countries": [], # ["EG", "NA", "MA", "ZA", "KE", "ET", "CG", "TZ", "GH", "TN", "NG"]
        "year": [], #2035, 2050
        "simpl": [],
        "clusters": [], # 10
        "ll": [], # copt
        "opts": [], # Co2L0.63
        "sopts": [], # 3H
        "discountrate": [], # 0.082
        "demand": [], # AB
        "h2export": [] # 3.33
    }
    for country in all_countries
}

for country, config in all_configs.items():
    all_wildcards[country]["run_name"].append(run_name_prefix)
    all_wildcards[country]["countries"].extend(config["countries"])
    all_wildcards[country]["year"].extend(config["scenario"]["planning_horizons"])
    all_wildcards[country]["simpl"].extend(config["scenario"]["simpl"])
    all_wildcards[country]["clusters"].extend(config["scenario"]["clusters"])
    all_wildcards[country]["ll"].extend(config["scenario"]["ll"])
    all_wildcards[country]["opts"].extend(config["scenario"]["opts"])
    all_wildcards[country]["sopts"].extend(config["scenario"]["sopts"])
    all_wildcards[country]["discountrate"].extend(config["costs"]["discountrate"])
    all_wildcards[country]["demand"].extend(config["scenario"]["demand"])
    all_wildcards[country]["h2export"].extend(config["export"]["h2export"])

sdir = Path.cwd() / "results"/ f"{run_name_prefix}_summary_20250502"
sdir.mkdir(exist_ok=True, parents=True)

In [37]:
all_wildcards["MA"]

{'run_name': ['H2G_B_250507'],
 'countries': ['MA'],
 'year': [2030],
 'simpl': [''],
 'clusters': ['3flex'],
 'll': ['copt'],
 'opts': ['Co2L'],
 'sopts': ['3h'],
 'discountrate': [0.071],
 'demand': ['AB'],
 'h2export': [10]}

In [38]:
# Collect all existing files from the directories
files_in_folder = {}
for country, path in all_postnetworks_dir.items():
    if path.exists() and path.is_dir():
        files_in_folder[f"{country}_{run_name_prefix}"] = list(path.glob('*'))  # Collect all files in the directory
    else:
        files_in_folder[f"{country}_{run_name_prefix}"] = []  # If the path doesn't exist or isn't a directory

# Print the collected files for each country
for key, files in files_in_folder.items():
    print(f"Files in {key}:")
    for file in files:
        print(f"  {file}")

Files in MA_H2G_B_250507:
  c:\Users\ljansen\projects\pypsa-earth-custom\results\MA_H2G_B_250507\postnetworks\elec_s_3flex_ec_lcopt_Co2L_3h_2030_0.071_AB_10export.nc
  c:\Users\ljansen\projects\pypsa-earth-custom\results\MA_H2G_B_250507\postnetworks\elec_s_3flex_ec_lcopt_Co2L_3h_2030_0.071_AB_13.33export.nc
  c:\Users\ljansen\projects\pypsa-earth-custom\results\MA_H2G_B_250507\postnetworks\elec_s_3flex_ec_lcopt_Co2L_3h_2030_0.071_AB_23.33export.nc
  c:\Users\ljansen\projects\pypsa-earth-custom\results\MA_H2G_B_250507\postnetworks\elec_s_3flex_ec_lcopt_Co2L_3h_2030_0.071_AB_3.33export.nc


In [39]:
cols = ["run_name", "country", "year", "simpl", "clusters", "ll", "opts", "sopts", "discountrate", "demand", "h2export"]

nc_files = pd.DataFrame(columns=cols + ["file"]).set_index(cols)

for country in all_countries:
    wc_keys = all_wildcards[country].keys()
    wc_values_combinations = list(product(*all_wildcards[country].values()))

    for combination in wc_values_combinations:
        wc = dict(zip(wc_keys, combination))
        for file in files_in_folder[f"{wc["countries"]}_{run_name_prefix}"]:
            if f"elec_s{wc['simpl']}_{wc['clusters']}_ec_l{wc['ll']}_{wc['opts']}_{wc['sopts']}_{wc['year']}_{wc['discountrate']}_{wc['demand']}_{wc['h2export']}export.nc" in file.name:
                nc_files.at[combination, "file"] = file
                print("File found:", file.name, "for:", f"{wc["countries"]}_{run_name_prefix}")
            else:
                print("File not found:", file.name, "for:", f"{wc["countries"]}_{run_name_prefix}")
                continue

File found: elec_s_3flex_ec_lcopt_Co2L_3h_2030_0.071_AB_10export.nc for: MA_H2G_B_250507
File not found: elec_s_3flex_ec_lcopt_Co2L_3h_2030_0.071_AB_13.33export.nc for: MA_H2G_B_250507
File not found: elec_s_3flex_ec_lcopt_Co2L_3h_2030_0.071_AB_23.33export.nc for: MA_H2G_B_250507
File not found: elec_s_3flex_ec_lcopt_Co2L_3h_2030_0.071_AB_3.33export.nc for: MA_H2G_B_250507


In [40]:
# initialise the dict of dataframes with the bus_carrier or other keys of interest

balances = init_stats_dict(nc_files, keys=[
    "AC", "H2", "oil", "gas", "co2 stored", 
    #"freshwater"
    ], name="bus_carrier")

optimal_capacities = init_stats_dict(nc_files, keys=["AC", "H2"], name="bus_carrier")

costs = init_stats_dict(nc_files, keys=["capex", "opex"], name="costs")


mean_marginal_prices = pd.DataFrame(index=nc_files.index, columns=["H2 export bus"])


for idx in nc_files.index:
    
    n = pypsa.Network(nc_files.at[idx,"file"])

    n.statistics.set_parameters(nice_names=False, drop_zero=False, round=6)

    # energy and mass market balances per bus_carrier in TWh

    for bus_carrier in balances.keys():

        ds = n.statistics.energy_balance(bus_carrier=bus_carrier, drop_zero=True).dropna().div(1e6).round(1).groupby("carrier").sum()

        for key,value in ds.items():
            balances[bus_carrier].at[idx,key] = value

        if bus_carrier == "AC" and "low voltage" in n.buses.carrier.unique():

            ds = n.statistics.energy_balance(bus_carrier="low voltage", drop_zero=True).dropna().div(1e6).round(1).groupby("carrier").sum()

            for key,value in ds.items():
                balances[bus_carrier].at[idx,key] = value

            balances[bus_carrier] = balances[bus_carrier].drop("electricity distribution grid", axis=1)  # drop energy between AC and distribution grid 

        #TODO: rename load carrier string of H2 export in the network from H2 to H2 export


    # optimal production capacity per bus_carrier in GW

    for bus_carrier in optimal_capacities.keys():

        ds = n.statistics.optimal_capacity(comps=["Generator", "Link", "StorageUnit"], bus_carrier=bus_carrier).dropna().groupby("carrier").sum().div(1e3).round(1)
        ds = ds[ds > 0]

        # if bus_carrier == "H2":
        #     ds = ds/n.links.groupby("carrier").mean(numeric_only=True).efficiency

        for key, value in ds.items():
            optimal_capacities[bus_carrier].at[idx, key] = value

        if bus_carrier == "AC" and "low voltage" in n.buses.carrier.unique():

            ds = n.statistics.optimal_capacity(comps=["Generator", "Link", "StorageUnit"], bus_carrier=bus_carrier).dropna().groupby("carrier").sum().div(1e3).round(1)
            ds = ds[ds>0]

            for key,value in ds.items():
                optimal_capacities[bus_carrier].at[idx,key] = value

    # costs per carrier

    ds = n.statistics.capex(nice_names=True, drop_zero=True).dropna().groupby("carrier").sum().div(1e9).round(4)

    for key,value in ds.items():
        costs["capex"].at[idx,key] = value
        
    ds = n.statistics.opex(nice_names=True, drop_zero=True).dropna().groupby("carrier").sum().div(1e9).round(4)

    for key,value in ds.items():
        costs["opex"].at[idx,key] = value
    
    # ASSUMPTIONS: assume marginal costs of last unit can be earned as export price for all export
    # adding those revenues for export as negative opex costs
    H2_export_price = n.buses_t.marginal_price["H2 export bus"].mean() # NB: hourly pattern is interesting!
    costs["opex"].at[idx,"H2 export"] = -(
        n.loads_t.p_set["H2 export load"].mul(n.buses_t.marginal_price["H2 export bus"]).sum()/1e9
    )

        
    # mean marginal prices per bus_carrier in EUR/MWh
    # TODO: use volume weighted average of marginal prices
    value = n.buses_t.marginal_price["H2 export bus"].mean() # NB: hourly pattern is interesting! 
    mean_marginal_prices.at[idx,"H2 export bus"] = value

    h2_buses = n.buses.loc[n.buses.index.str.contains("H2")]
    for bus in h2_buses.index:
        value = n.buses_t.marginal_price[bus].mean() # NB: hourly pattern is interesting! 
        mean_marginal_prices.at[idx, bus] = value
    # value = n.buses_t.marginal_price["H2 export bus"].mean() # NB: hourly pattern is interesting! 
    # mean_marginal_prices.at[idx,"H2 export bus"] = value

INFO:pypsa.io:Imported network elec_s_3flex_ec_lcopt_Co2L_3h_2030_0.071_AB_10export.nc has buses, carriers, generators, global_constraints, lines, links, loads, storage_units, stores


In [41]:
print(mean_marginal_prices["H2 export bus"]) #NB: all other H2 nodal prices are higher

run_name      country  year  simpl  clusters  ll    opts  sopts  discountrate  demand  h2export
H2G_B_250507  MA       2030         3flex     copt  Co2L  3h     0.071         AB      10          42.601143
Name: H2 export bus, dtype: object


In [42]:
# simplify index for plotting: define a scen_str column and drop all other columns
#NB: the following is specific to the experiment

index_levels_to_drop=["simpl", "clusters", "ll", "sopts", "discountrate"]

for key in balances.keys():
    balances[key] = drop_index_levels(balances[key], to_drop=index_levels_to_drop)

for key in optimal_capacities.keys():
    optimal_capacities[key] = drop_index_levels(optimal_capacities[key], to_drop=index_levels_to_drop)
    
for key in costs.keys():
    costs[key] = drop_index_levels(costs[key], to_drop=index_levels_to_drop)

Dropping index level simpl with only one unique value: 
Dropping index level clusters with only one unique value: 3flex
Dropping index level ll with only one unique value: copt
Dropping index level sopts with only one unique value: 3h
Dropping index level discountrate with only one unique value: 0.071
Dropping index level simpl with only one unique value: 
Dropping index level clusters with only one unique value: 3flex
Dropping index level ll with only one unique value: copt
Dropping index level sopts with only one unique value: 3h
Dropping index level discountrate with only one unique value: 0.071
Dropping index level simpl with only one unique value: 
Dropping index level clusters with only one unique value: 3flex
Dropping index level ll with only one unique value: copt
Dropping index level sopts with only one unique value: 3h
Dropping index level discountrate with only one unique value: 0.071
Dropping index level simpl with only one unique value: 
Dropping index level clusters with 

# Plotting

In [43]:
all_wildcards

{'MA': {'run_name': ['H2G_B_250507'],
  'countries': ['MA'],
  'year': [2030],
  'simpl': [''],
  'clusters': ['3flex'],
  'll': ['copt'],
  'opts': ['Co2L'],
  'sopts': ['3h'],
  'discountrate': [0.071],
  'demand': ['AB'],
  'h2export': [10]}}

In [65]:
# select and define index group (multiindex) for plots: which solutions to show at once
print(balances["AC"].index.names)

#idx_group = balances["AC"].index
idx_group = idx_slice[["H2G_B_250507"],["MA"],:,:] 
idx_group_name = "EG_H2export_scenarios"

print(idx_group_name) 
print(idx_group)

scen_order = [] #'3.33MtH2export', '13.33MtH2export', '23.33MtH2export',

['run_name', 'country', 'year', 'opts', 'demand', 'h2export']
EG_H2export_scenarios
(['H2G_B_250507'], ['MA'], slice(None, None, None), slice(None, None, None))


In [66]:
width = 700
heights = {"1": 300,"2": 600,"3": 700,}

fig_kwargs = dict(
    x="scen", 
    y="value", 
    color="variable", 
    facet_col="year", 
    barmode="relative", 
    text="value", 
    #facet_col_wrap=1, facet_col_spacing=0.07,
    width=width, 
    height=heights["2"],
    text_auto =".0f",
)



## Energy market balance | Electricity

In [67]:
df = prepare_dataframe(balances["AC"], idx_group)

#df.variable = df.variable.map(rename_electricity)
# supply_df = df[df["value"]>=0.01].copy()
# supply_df = supply_df.groupby(["scen","year","variable"], as_index=False).sum()

# #supply_df = supply_df.query("variable not in ['Battery', 'Hydro Power']")
# supply_sums_df = supply_df.groupby(["scen","year"]).sum(numeric_only=True).round(0)
(supply_df, supply_sum_df, demand_df, demand_sum_df) = get_supply_demand_from_balance(df)

In [68]:
df.scen.values

array(['Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB',
       'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB',
       'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB',
       'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB',
       'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB',
       'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB',
       'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB', 'Co2L-AB'],
      dtype=object)

In [69]:
fig = px.bar(
    supply_df, 
    **fig_kwargs,
    labels={"value":"Electricity supply in TWh<sub>el</sub>",
            "year":"",
            "variable":"", 
            "scen":""
            },
    #color_discrete_map=colors["electricity"],
    category_orders={
        #"variable":list(colors["electricity"].keys())
        "scen": scen_order
        },
    #facet_col_wrap=1, facet_col_spacing=0.07,
)
fig.update_layout(legend_traceorder="reversed",)
update_layout(fig)

print("Totals per scenario and year:", supply_sum_df)
fig.show()
#save_plotly_fig(supply_df, fig, sdir, f"{idx_group_name}_electricity_supply_TWh")

Totals per scenario and year:               value
scen    year       
Co2L-AB 2030   82.0


In [70]:
fig = px.bar(
    demand_df, 
    **fig_kwargs,
    #color_discrete_map=colors["electricity"],
    category_orders={
        #"variable":list(colors["electricity"].keys())
        "scen": scen_order
        },
    labels={"value":"Electricity demand in TWh<sub>el</sub>","year":"",
            "variable":"", 
            "scen":""},
)
fig.update_layout(legend_traceorder="reversed",)
update_layout(fig)

print("Totals per scenario and year:", demand_sum_df)
fig.show()
#save_plotly_fig(demand_df, fig, sdir, f"{idx_group_name}_electricity_demand_TWh")

Totals per scenario and year:               value
scen    year       
Co2L-AB 2030   82.0


## Energy and feedstock market balance | Hydrogen

In [71]:
df = prepare_dataframe(balances["H2"], idx_group)
df = df[df["value"]!=0]
#df.variable = df.variable.map(rename_h2)

(supply_df, supply_sum_df, demand_df, demand_sum_df) = get_supply_demand_from_balance(df)

In [72]:
fig = px.bar(
    df, 
    **fig_kwargs,
    #color_discrete_map=colors["hydrogen"],
    category_orders={
        #"variable":list(colors["hydrogen"].keys())
        "scen": scen_order
        },
    #facet_col_spacing=0.1,
    labels={"value":"H2 balance in TWh","year":"",
            "variable":"", 
            "scen": ""},

)
update_layout(fig)

print("Totals per scenario and year:", demand_sum_df)
fig.show()
#save_plotly_fig(df, fig, fig, sdir, f"{idx_group_name}_hydrogen_balance_TWh")

Totals per scenario and year:               value
scen    year       
Co2L-AB 2030   10.0


## Energy and feedstock market balance | Liquid fuel

In [73]:
df = prepare_dataframe(balances["oil"], idx_group)
df = df[df["value"]!=0]
#df.variable = df.variable.map(rename_oil)

(supply_df, supply_sum_df, demand_df, demand_sum_df) = get_supply_demand_from_balance(df)

In [74]:
fig = px.bar(
    df, 
    **fig_kwargs,
    #color_discrete_map=colors["oil"],
    category_orders={
        #"variable":list(colors["oil"].keys())
        "scen": scen_order
        },
    #facet_col_spacing=0.1,
    labels={"value":"Liquid fuel balance in TWh","year":"",
            "variable":"", 
            "scen":""},

)
update_layout(fig)

print("Totals per scenario and year:", demand_sum_df)
fig.show()
#save_plotly_fig(df, fig, fig, sdir, f"{idx_group_name}_liquid_fuel_balance_TWh")

Totals per scenario and year:               value
scen    year       
Co2L-AB 2030   98.0


## Energy and feedstock market balance | CH4

In [75]:
df = prepare_dataframe(balances["gas"], idx_group)
df = df[df["value"]!=0]
#df.variable = df.variable.map(rename_gas)

(supply_df, supply_sum_df, demand_df, demand_sum_df) = get_supply_demand_from_balance(df)

In [76]:
fig = px.bar(
    df, 
    **fig_kwargs,
    #color_discrete_map=colors["gas"],
    category_orders={
        #"variable":list(colors["gas"].keys())
        "scen": scen_order
        },
    #facet_col_spacing=0.1,
    labels={"value":"CH4 balance in TWh","year":"",
            "variable":"", 
            "scen":""},

)
update_layout(fig)

print("Totals per scenario and year:", demand_sum_df)
fig.show()
#save_plotly_fig(df, fig, fig, sdir, f"{idx_group_name}_CH4_balance_TWh")

Totals per scenario and year:               value
scen    year       
Co2L-AB 2030   19.0


## Energy and feedstock market balance | CO2

In [56]:
df = prepare_dataframe(balances["co2 stored"], idx_group)
df = df[df["value"]!=0]

#df.variable = df.variable.map(rename_co2)

(supply_df, supply_sum_df, demand_df, demand_sum_df) = get_supply_demand_from_balance(df)

In [57]:
fig = px.bar(
    df, 
    **fig_kwargs,
    #color_discrete_map=colors["co2"],
    category_orders={
        #"variable":list(colors["gas"].keys())
        "scen": scen_order
        },
    #facet_col_spacing=0.1,
    labels={"value":"CO<sub>2</sub> Capture (+) and Usage (-) in Mt","year":"","variable":"", "scen":""},
)
update_layout(fig)

print("Totals per scenario and year:", demand_sum_df)
fig.show()
#save_plotly_fig(df, fig, fig, sdir, f"{idx_group_name}_CO2_capture_and_usage_Mt")

Totals per scenario and year: Empty DataFrame
Columns: [value]
Index: []


## Installed Capacities | Electricity

In [58]:
df = prepare_dataframe(optimal_capacities["AC"], idx_group)
df = df[df["value"]<1e6] #drop loadshedding capacity
#df.variable = df.variable.map(rename_electricity)

(supply_df, supply_sum_df, demand_df, demand_sum_df) = get_supply_demand_from_balance(df)

In [59]:
fig = px.bar(
    df, 
    **fig_kwargs,
    #barmode="stack",
    #color_discrete_map=colors["electricity"],
    category_orders={
        #"variable":list(colors["gas"].keys())
        "scen": scen_order
        },
    #facet_col_spacing=0.1,
    #range_y=(0,supply_sum_df.max().value*1.1),
    labels={"value":"Installed capacity in GW<sub>el</sub>","year":"",
            #"variable":"",
            "scen":""},
)
update_layout(fig)

print("Totals per scenario and year:", supply_sum_df)
fig.show()
#save_plotly_fig(df, fig, fig, sdir, f"{idx_group_name}_installed_capacity_AC_GW")

Totals per scenario and year:               value
scen    year       
Co2L-AB 2030   86.0


In [60]:
df = prepare_dataframe(optimal_capacities["H2"], idx_group)
df = df[df["value"]<1e6] #drop loadshedding capacity
#df.variable = df.variable.map(rename_H2)

(supply_df, supply_sum_df, demand_df, demand_sum_df) = get_supply_demand_from_balance(df)

In [61]:
df

,run_name,scen,year,country,variable,value
0,H2G_B_250507,Co2L-AB,2030,MA,SMR,1.2


In [62]:
fig = px.bar(
    df, 
    **fig_kwargs,
    color_discrete_map={"H2 Electrolysis":"#179c7d"},
    category_orders={
        #"variable":list(colors["gas"].keys())
        "scen": scen_order
        },
    labels={"value":"Installed capacity in GW<sub>H2</sub>","year":"","variable":"","scen":""},
)
update_layout(fig)

print("Totals per scenario and year:", supply_sum_df)
fig.show()
#save_plotly_fig(df, fig, fig, sdir, f"{idx_group_name}_installed_capacity_AC_GW")

Totals per scenario and year:               value
scen    year       
Co2L-AB 2030    1.0


In [63]:
stats_df = pd.concat([costs["capex"],costs["opex"]],axis=0)
df = prepare_dataframe(stats_df, idx_group)
df = df[df["value"]!=0] #drop loadshedding capacity
#df.variable = df.variable.map(rename_H2)

(supply_df, supply_sum_df, demand_df, demand_sum_df) = get_supply_demand_from_balance(df, threshold=0., round=1)

In [64]:
fig_kwargs["height"] = heights["3"]
fig = px.bar(
    df, 
    **fig_kwargs,
    #color_discrete_map=colors["costs"],
    category_orders={
        #"variable":list(colors["costs"].keys())
        "scen": scen_order
        },
    facet_col_spacing=0.1,
    labels={"value":"System costs in billion USD/year","year":"",
            #"variable":"", 
            "scen":""},
)
update_layout(fig)

print("Totals per scenario and year:", supply_sum_df.loc[scen_order])
fig.show()
#save_plotly_fig(df, fig, fig, sdir, f"{idx_group_name}_installed_capacity_AC_GW")

KeyError: "['3.33MtH2export' '13.33MtH2export' '23.33MtH2export'] not in index"

In [ ]:
df.

## Relative costs

## Total and relative CO2 emissions

### Water consumption

Marginal H2 production costs per H2 export volume
* x = H2 export
* y = Marginal H2 costs
* color = country
* facet_col = year
* markers = True

Dumbbell plot https://plotly.com/python/dumbbell-plots/
color = H2 export
y = country
x = marginal h2 cost







